# 02 — XGBoost Risk Model Training

**FreightIQ** uses an **XGBoost classifier** to score each shipment on a 0–1 delay probability scale. SHAP values provide human-readable explanations of *why* each shipment is flagged.

This notebook walks through:

1. **Data Loading** — Load the synthetic shipment dataset
2. **Exploratory Data Analysis** — Understand feature distributions and correlations
3. **Feature Engineering** — Prepare the feature matrix
4. **Model Training** — Train XGBoost with evaluation metrics
5. **Evaluation** — AUC, precision, recall, confusion matrix
6. **SHAP Explanations** — Interpret model decisions visually
7. **Inference** — Score new shipments using the `RiskScorer` class

---

### Prerequisites

- Synthetic data generated: `python scripts/generate_data.py`
- Dependencies installed: `pip install -r requirements.txt`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Use a clean style for all plots
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("✅ Setup complete")

---
## 1. Data Loading

The synthetic dataset is generated by `scripts/generate_data.py` using the **Faker** library. Each row represents a shipment with engineered risk features and a binary target (`is_delayed`).

In [ ]:
df = pd.read_csv("../data/synthetic/shipments.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Target variable distribution
print("Target distribution (is_delayed):")
print(df["is_delayed"].value_counts())
print(f"\nDelay rate: {df['is_delayed'].mean():.1%}")

---
## 2. Exploratory Data Analysis

Let's understand the feature distributions and their relationship to delays.

In [ ]:
from src.ml.feature_engineering import FEATURE_COLS

print(f"Feature columns ({len(FEATURE_COLS)}):")
for col in FEATURE_COLS:
    print(f"  • {col}")

df[FEATURE_COLS].describe().round(3)

In [ ]:
# Feature distributions: delayed vs on-time
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    df[df["is_delayed"] == 0][col].hist(ax=ax, bins=25, alpha=0.6, label="On Time", color="#22c55e")
    df[df["is_delayed"] == 1][col].hist(ax=ax, bins=25, alpha=0.6, label="Delayed", color="#ef4444")
    ax.set_title(col.replace("_", " ").title(), fontsize=10)
    ax.legend(fontsize=8)

# Hide unused subplot
if len(FEATURE_COLS) < len(axes):
    for j in range(len(FEATURE_COLS), len(axes)):
        axes[j].set_visible(False)

plt.suptitle("Feature Distributions: On Time vs Delayed", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr = df[FEATURE_COLS + ["is_delayed"]].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

labels = [c.replace("_", "\n") for c in corr.columns]
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)

# Annotate cells
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nCorrelation with target (is_delayed):")
print(corr["is_delayed"].drop("is_delayed").sort_values(ascending=False).to_string())

In [ ]:
# Average delay rate by region and carrier
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df.groupby("region")["is_delayed"].mean().sort_values().plot.barh(ax=ax1, color="#6366f1")
ax1.set_xlabel("Delay Rate")
ax1.set_title("Delay Rate by Region")

df.groupby("carrier")["is_delayed"].mean().sort_values().plot.barh(ax=ax2, color="#f59e0b")
ax2.set_xlabel("Delay Rate")
ax2.set_title("Delay Rate by Carrier")

plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

The `prepare_training_data()` function in `src/ml/feature_engineering.py` extracts the 7 numerical features and the target variable.

| Feature | Description |
|---|---|
| `carrier_reliability` | Historical on-time rate (0–1) |
| `region_disruption_count` | Active disruptions in region |
| `days_to_delivery` | Days until expected delivery |
| `weather_severity` | Max weather severity on route (0–1) |
| `route_risk_score` | Historical delay rate for route |
| `cargo_value_usd` | Cargo value in USD |
| `news_sentiment_score` | Aggregated news sentiment for region |

In [ ]:
from src.ml.feature_engineering import prepare_training_data

X, y = prepare_training_data(df)

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape:         {y.shape}")
print(f"Target distribution:  {y.value_counts().to_dict()}")
print(f"\nFeature columns: {list(X.columns)}")
X.head()

---
## 4. Model Training

We use `ModelTrainer` from `src/ml/train.py` which:
- Splits data 80/20 with stratification
- Trains an XGBoost classifier (100 trees, depth 4, learning rate 0.1)
- Returns AUC, precision, and recall metrics

In [ ]:
from src.ml.train import ModelTrainer

trainer = ModelTrainer(random_state=42)

print("Training XGBoost risk model...")
metrics = trainer.train(X, y)

print(f"\n{'='*40}")
print(f"  Train AUC:  {metrics['train_auc']:.3f}")
print(f"  Val AUC:    {metrics['val_auc']:.3f}")
print(f"  Precision:  {metrics['precision']:.3f}")
print(f"  Recall:     {metrics['recall']:.3f}")
print(f"{'='*40}")

In [ ]:
# XGBoost built-in feature importance
import xgboost as xgb

fig, ax = plt.subplots(figsize=(8, 5))
xgb.plot_importance(
    trainer.model, 
    ax=ax, 
    importance_type="gain",
    title="Feature Importance (Gain)",
    color="#6366f1"
)
plt.tight_layout()
plt.show()

---
## 5. Evaluation

Let's look at the model performance in more detail — ROC curve, confusion matrix, and score distribution.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

# Reproduce the same split used in training
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_pred_proba = trainer.model.predict_proba(X_val)[:, 1]
y_pred = trainer.model.predict(X_val)

print(classification_report(y_val, y_pred, target_names=["On Time", "Delayed"]))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax1.plot(fpr, tpr, color="#6366f1", lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#94a3b8", lw=1)
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curve", fontweight="bold")
ax1.legend(loc="lower right")
ax1.set_xlim([-0.02, 1.02])
ax1.set_ylim([-0.02, 1.02])

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
im = ax2.imshow(cm, cmap="Blues")
ax2.set_xticks([0, 1])
ax2.set_xticklabels(["On Time", "Delayed"])
ax2.set_yticks([0, 1])
ax2.set_yticklabels(["On Time", "Delayed"])
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")
ax2.set_title("Confusion Matrix", fontweight="bold")
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16, fontweight="bold")
plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Score distribution
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(y_pred_proba[y_val == 0], bins=30, alpha=0.6, label="On Time", color="#22c55e")
ax.hist(y_pred_proba[y_val == 1], bins=30, alpha=0.6, label="Delayed", color="#ef4444")
ax.axvline(x=0.5, color="#1e293b", linestyle="--", lw=1.5, label="Threshold (0.5)")
ax.axvline(x=0.4, color="#f59e0b", linestyle=":", lw=1.5, label="Medium threshold (0.4)")
ax.axvline(x=0.7, color="#ef4444", linestyle=":", lw=1.5, label="High threshold (0.7)")

ax.set_xlabel("Predicted Risk Score")
ax.set_ylabel("Count")
ax.set_title("Risk Score Distribution by Actual Outcome", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

from src.ml.utils import risk_level_from_score
levels = pd.Series(y_pred_proba).apply(risk_level_from_score)
print("Risk level distribution on validation set:")
print(levels.value_counts().to_string())

---
## 6. SHAP Explanations

SHAP (SHapley Additive exPlanations) values tell us *how much each feature contributed* to a specific prediction. This is what powers the human-readable explanations in FreightIQ's alerts.

The `RiskScorer._generate_explanation()` method converts these values into plain English.

In [ ]:
import shap

# Initialize the SHAP explainer for our trained model
explainer = shap.TreeExplainer(trainer.model)

# Compute SHAP values for the validation set
shap_values = explainer.shap_values(X_val)

print(f"SHAP values shape: {shap_values.shape}")
print(f"(samples × features)")

In [ ]:
# Global feature importance via SHAP
shap.summary_plot(shap_values, X_val, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (mean |SHAP value|)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm plot — shows direction of feature impact
shap.summary_plot(shap_values, X_val, show=False)
plt.title("SHAP Beeswarm Plot", fontweight="bold")
plt.tight_layout()
plt.show()

print("📌 Red = high feature value, Blue = low feature value")
print("   Dots to the right push risk UP, dots to the left push risk DOWN")

In [ ]:
# Explain a single high-risk prediction
high_risk_idx = y_pred_proba.argmax()
print(f"Highest risk shipment (index {high_risk_idx}):")
print(f"  Risk score: {y_pred_proba[high_risk_idx]:.3f}")
print(f"  Actual:     {'Delayed' if y_val.iloc[high_risk_idx] == 1 else 'On Time'}")
print(f"  Features:")
for col in FEATURE_COLS:
    print(f"    {col:30s} = {X_val.iloc[high_risk_idx][col]}")

# Waterfall plot for this prediction
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[high_risk_idx],
        base_values=explainer.expected_value,
        data=X_val.iloc[high_risk_idx],
        feature_names=FEATURE_COLS
    ),
    show=False
)
plt.title("SHAP Waterfall — Highest Risk Shipment", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Explain a low-risk prediction for comparison
low_risk_idx = y_pred_proba.argmin()
print(f"Lowest risk shipment (index {low_risk_idx}):")
print(f"  Risk score: {y_pred_proba[low_risk_idx]:.3f}")
print(f"  Actual:     {'Delayed' if y_val.iloc[low_risk_idx] == 1 else 'On Time'}")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[low_risk_idx],
        base_values=explainer.expected_value,
        data=X_val.iloc[low_risk_idx],
        feature_names=FEATURE_COLS
    ),
    show=False
)
plt.title("SHAP Waterfall — Lowest Risk Shipment", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Inference with `RiskScorer`

The `RiskScorer` class in `src/ml/scorer.py` is what the FastAPI backend and LangGraph agent use at runtime. It wraps the trained model with SHAP-based explanations.

Let's demonstrate scoring individual shipments.

In [ ]:
# First, save the model we just trained so RiskScorer can load it
model_path = trainer.save("../data/models/")
print(f"Model saved to: {model_path}")

In [ ]:
from src.ml.scorer import RiskScorer

scorer = RiskScorer("../data/models/xgboost_risk.json")
print("✅ RiskScorer loaded")

In [ ]:
# Score a single high-risk shipment
high_risk_shipment = {
    "carrier_reliability": 0.62,
    "region_disruption_count": 4,
    "days_to_delivery": 3,
    "weather_severity": 0.85,
    "route_risk_score": 0.78,
    "cargo_value_usd": 350_000,
    "news_sentiment_score": -0.6,
}

result = scorer.score(high_risk_shipment)
print("High-risk shipment scoring:")
print(f"  Risk score: {result['risk_score']}")
print(f"  Risk level: {result['risk_level']}")
print(f"  {result['explanation']}")

In [ ]:
# Score a low-risk shipment
low_risk_shipment = {
    "carrier_reliability": 0.96,
    "region_disruption_count": 0,
    "days_to_delivery": 20,
    "weather_severity": 0.1,
    "route_risk_score": 0.15,
    "cargo_value_usd": 45_000,
    "news_sentiment_score": 0.7,
}

result = scorer.score(low_risk_shipment)
print("Low-risk shipment scoring:")
print(f"  Risk score: {result['risk_score']}")
print(f"  Risk level: {result['risk_level']}")
print(f"  {result['explanation']}")

In [ ]:
# Batch scoring — score an entire DataFrame at once
batch_df = pd.read_csv("../data/synthetic/shipments.csv")
batch_results = scorer.score(batch_df)

# Add scores back to the dataframe for analysis
scores_df = batch_df.copy()
scores_df["risk_score"] = [r["risk_score"] for r in batch_results]
scores_df["risk_level"] = [r["risk_level"] for r in batch_results]

print(f"Scored {len(scores_df)} shipments\n")
print("Risk level distribution:")
print(scores_df["risk_level"].value_counts().to_string())
print(f"\nAverage risk score: {scores_df['risk_score'].mean():.3f}")

# Show top 5 highest risk shipments
print("\n--- Top 5 Highest Risk Shipments ---")
top5 = scores_df.nlargest(5, "risk_score")[["shipment_id", "route", "carrier", "risk_score", "risk_level"]]
print(top5.to_string(index=False))

In [ ]:
# Visualize risk distribution
from src.ml.utils import risk_color

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of risk scores
ax1.hist(scores_df["risk_score"], bins=30, color="#6366f1", edgecolor="white", alpha=0.8)
ax1.axvline(x=0.4, color="#f59e0b", linestyle="--", lw=2, label="Medium threshold")
ax1.axvline(x=0.7, color="#ef4444", linestyle="--", lw=2, label="High threshold")
ax1.set_xlabel("Risk Score")
ax1.set_ylabel("Count")
ax1.set_title("Risk Score Distribution (All Shipments)", fontweight="bold")
ax1.legend()

# Risk level pie chart
level_counts = scores_df["risk_level"].value_counts()
colors = [risk_color(level) for level in level_counts.index]
ax2.pie(level_counts, labels=level_counts.index, colors=colors, autopct="%1.1f%%",
        startangle=90, textprops={"fontsize": 12})
ax2.set_title("Risk Level Distribution", fontweight="bold")

plt.tight_layout()
plt.show()

---
## Key Takeaways

| Design Decision | Choice | Why |
|---|---|---|
| Model | XGBoost Classifier | Fast, interpretable, works well on tabular data |
| Trees / Depth | 100 trees, depth 4 | Balanced complexity — avoids overfitting on synthetic data |
| Explainability | SHAP `TreeExplainer` | Exact SHAP values for tree models (fast), human-readable output |
| Risk thresholds | LOW < 0.4 < MEDIUM < 0.7 < HIGH | Calibrated for action priority in operations |
| Training data | Synthetic (Faker) | Enables demo without proprietary data; same pipeline works on real data |

### Next Steps

- See `01_rag_exploration.ipynb` for the RAG pipeline
- See `src/agent/nodes.py` → `score_node()` for how the agent calls `RiskScorer` at runtime
- See `src/api/routes/shipments.py` for the FastAPI endpoint that serves risk scores